# 🔧 Notebook 02 — Feature Engineering
**SME Retail | Inventory-Demand Co-optimization**

> เป้าหมาย: สร้าง feature matrix พร้อม train สำหรับ M1 Demand Forecast และ M2 Lead Time Model

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = "data/raw/"
sales     = pd.read_csv(DATA_PATH + "sales_transaction.csv",  parse_dates=["datetime"])
po        = pd.read_csv(DATA_PATH + "purchasing_order.csv",   parse_dates=["po_date","arrival_date","expire_date"])
products  = pd.read_csv(DATA_PATH + "product_master.csv")
promos    = pd.read_csv(DATA_PATH + "promotion_master.csv",   parse_dates=["start_date","end_date"])
stores    = pd.read_csv(DATA_PATH + "store_master.csv")
print("✅ Data loaded")

## 1. Weekly Aggregation (Target Variable)

In [ ]:
# aggregate sales → weekly per (store, product)
sales['revenue'] = sales['price'] * sales['qty']
sales['week']    = sales['datetime'].dt.to_period('W').dt.start_time

weekly = (
    sales.groupby(['week','store_id','product_id'])
    .agg(qty_sold=('qty','sum'), revenue=('revenue','sum'), n_transactions=('qty','count'))
    .reset_index()
)

print(f"Weekly agg shape: {weekly.shape}")
print(f"Date range: {weekly['week'].min().date()} → {weekly['week'].max().date()}")
weekly.head(8)

## 2. Promotion Feature Join

In [ ]:
# สร้าง weekly promotion calendar
# สำหรับแต่ละ (product_id, week) → มีโปรโมชั่นไหม? discount เท่าไหร่?

promo_weeks = []
for _, row in promos.iterrows():
    weeks = pd.date_range(row['start_date'], row['end_date'], freq='W-MON')
    for w in weeks:
        promo_weeks.append({
            'product_id':   row['product_id'],
            'week':         w,
            'promotion_id': row['promotion_id'],
            'discount':     row['discount'],
        })

promo_cal = pd.DataFrame(promo_weeks)

# join กับ weekly data
weekly = weekly.merge(promo_cal, on=['week','product_id'], how='left')
weekly['has_promo']   = weekly['discount'].notna().astype(int)
weekly['discount']    = weekly['discount'].fillna(0)

# next week promotion flag (สำคัญมาก — ถ้าสัปดาห์หน้ามีโปรฯ ต้องสต็อกเพิ่ม)
promo_next = promo_cal.copy()
promo_next['week'] = promo_next['week'] - pd.Timedelta(weeks=1)
promo_next = promo_next[['product_id','week']].drop_duplicates()
promo_next['has_promo_next_week'] = 1

weekly = weekly.merge(promo_next, on=['week','product_id'], how='left')
weekly['has_promo_next_week'] = weekly['has_promo_next_week'].fillna(0).astype(int)

print(f"Promo coverage: {weekly['has_promo'].mean()*100:.1f}% of rows")
print(f"Next-week promo: {weekly['has_promo_next_week'].mean()*100:.1f}% of rows")

## 3. Lag & Rolling Features

In [ ]:
# sort ก่อนเสมอ — critical สำหรับ time series
weekly = weekly.sort_values(['store_id','product_id','week']).reset_index(drop=True)

def add_lag_rolling(df, group_cols, target_col):
    df = df.copy()
    grp = df.groupby(group_cols)[target_col]

    # Lag features
    for lag in [1, 2, 4, 8]:
        df[f'lag_{lag}w'] = grp.shift(lag)

    # Rolling mean & std
    for window in [4, 8]:
        df[f'rolling_mean_{window}w'] = grp.shift(1).rolling(window).mean().reset_index(level=list(range(len(group_cols))), drop=True)
        df[f'rolling_std_{window}w']  = grp.shift(1).rolling(window).std().reset_index(level=list(range(len(group_cols))), drop=True)

    return df

weekly = add_lag_rolling(weekly, ['store_id','product_id'], 'qty_sold')
print("Lag & rolling features added")
print(f"Feature columns: {[c for c in weekly.columns if 'lag' in c or 'rolling' in c]}")

## 4. Temporal & Context Features

In [ ]:
# Temporal features
weekly['week_of_year']  = weekly['week'].dt.isocalendar().week.astype(int)
weekly['month']         = weekly['week'].dt.month
weekly['quarter']       = weekly['week'].dt.quarter
weekly['is_month_end']  = (weekly['week'].dt.day >= 24).astype(int)

# Product features
weekly = weekly.merge(products[['product_id','product_taxonomies','price']], on='product_id', how='left')
# Label encode category
cat_map = {c: i for i, c in enumerate(products['product_taxonomies'].unique())}
weekly['product_cat_enc'] = weekly['product_taxonomies'].map(cat_map)

# Store features
weekly = weekly.merge(stores, on='store_id', how='left')
store_map = {c: i for i, c in enumerate(stores['store_taxonomies'].unique())}
weekly['store_type_enc'] = weekly['store_taxonomies'].map(store_map)

# Price elasticity proxy
avg_cat_price = weekly.groupby('product_taxonomies')['price'].transform('mean')
weekly['price_vs_category'] = weekly['price'] / avg_cat_price

print("Temporal & context features added ✅")
print(f"Total features so far: {weekly.shape[1]} columns")

## 5. Feature Matrix — Final

In [ ]:
FEATURE_COLS = [
    # Lag
    'lag_1w','lag_2w','lag_4w','lag_8w',
    # Rolling
    'rolling_mean_4w','rolling_std_4w','rolling_mean_8w','rolling_std_8w',
    # Promotion
    'has_promo','discount','has_promo_next_week',
    # Temporal
    'week_of_year','month','quarter','is_month_end',
    # Context
    'product_cat_enc','store_type_enc','price_vs_category',
]
TARGET_COL = 'qty_sold'

# Drop rows where lag features are NaN (first 8 weeks per group)
feature_df = weekly.dropna(subset=['lag_8w']).reset_index(drop=True)

print(f"Feature matrix shape: {feature_df.shape}")
print(f"\nFeature columns ({len(FEATURE_COLS)}):")
for f in FEATURE_COLS:
    null_pct = feature_df[f].isnull().mean() * 100
    print(f"  {f:<30} null: {null_pct:.1f}%")

## 6. Train / Val / Test Split (Time-based)

In [ ]:
# Time-based split — NEVER random สำหรับ time series
TRAIN_END = '2024-09-30'
VAL_END   = '2024-10-31'
# TEST      = Nov–Dec 2024

train = feature_df[feature_df['week'] <= TRAIN_END]
val   = feature_df[(feature_df['week'] > TRAIN_END) & (feature_df['week'] <= VAL_END)]
test  = feature_df[feature_df['week'] > VAL_END]

X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_val,   y_val   = val[FEATURE_COLS],   val[TARGET_COL]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET_COL]

print(f"Train : {len(train):>5} rows | {train['week'].min().date()} → {train['week'].max().date()}")
print(f"Val   : {len(val):>5} rows | {val['week'].min().date()} → {val['week'].max().date()}")
print(f"Test  : {len(test):>5} rows | {test['week'].min().date()} → {test['week'].max().date()}")

# Save feature store
feature_df.to_csv("data/processed/feature_store.csv", index=False)
print("\n✅ Feature store saved → data/processed/feature_store.csv")

## 7. Lead Time Feature Matrix (M2)

In [ ]:
# Feature matrix สำหรับ M2 Lead Time Model
po['lead_time_days'] = (po['arrival_date'] - po['po_date']).dt.days
po = po.merge(products[['product_id','product_taxonomies']], on='product_id', how='left')

po['po_month']  = po['po_date'].dt.month
po['po_dow']    = po['po_date'].dt.dayofweek
po['po_qty_log']= np.log1p(po['qty'])

lt_cat_map = {c: i for i, c in enumerate(po['product_taxonomies'].dropna().unique())}
po['product_cat_enc'] = po['product_taxonomies'].map(lt_cat_map)

wh_map = {w: i for i, w in enumerate(po['warehouse_id'].unique())}
po['warehouse_enc'] = po['warehouse_id'].map(wh_map)

LT_FEATURES = ['po_month','po_dow','po_qty_log','product_cat_enc','warehouse_enc']
LT_TARGET   = 'lead_time_days'

lt_df = po.dropna(subset=LT_FEATURES + [LT_TARGET])
print(f"Lead time feature matrix: {lt_df.shape}")
lt_df[LT_FEATURES + [LT_TARGET]].describe().round(2)